In [ ]:
%cd ../

In [ ]:
from pathlib import Path

import pandas as pd
import psycopg as pg
from loguru import logger
from psycopg import sql
from psycopg.rows import dict_row

In [ ]:
DB_PORT=''
DB_PWD=''
DB_USER=''
DB_NAME=''
DB_HOST=''

# Upload dim tables

### `dim_restaurants`

In [ ]:
path = "data/processed/dim_restaurants.xlsx"
dim_restaurants = pd.read_excel(path)
dim_restaurants.head()

In [ ]:
values = [
    (r.restaurant_id, r.restaurant, r.restaurant_short)
    for r in dim_restaurants.itertuples()
]

In [ ]:
# Compose SQL
cols = ['restaurant_id', 'restaurant', 'restaurant_short']
table = 'dim_restaurants'

try:
    with pg.connect(
        user=DB_USER,
        password=DB_PWD,
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        row_factory=dict_row,
    ) as conn:
        with conn.cursor() as cur:
            query = """
                insert into {table}
                    ({cols})
                values ({values})
                ;
            """

            stmt = (
                sql
                .SQL(query)
                .format(
                    table=sql.Identifier(table),
                    cols=sql.SQL(', ').join(map(sql.Identifier, cols)),
                    values=sql.SQL(', ').join(sql.Placeholder() * len(values[0]))
                )
            )

            # for v in values:
            cur.executemany(stmt, values)
            conn.commit()

            # ret = cur.fetchall()
            

except pg.OperationalError as e:
    logger.error(f"Connect to DB got error: {e}")

### `dim_meal_types`

In [ ]:
path = "data/processed/dim_meal_types.xlsx"
dim_meal_types = pd.read_excel(path)
dim_meal_types.head()

In [ ]:
values = [
    (r.meal_type_id, r.meal_type, r.meal_type_en)
    for r in dim_meal_types.itertuples()
]

values[:2]

In [ ]:
# Compose SQL
cols = ['meal_type_id', 'meal_type', 'meal_type_en']
table = 'dim_meal_types'

try:
    with pg.connect(
        user=DB_USER,
        password=DB_PWD,
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        row_factory=dict_row,
    ) as conn:
        with conn.cursor() as cur:
            query = """
                insert into {table}
                    ({cols})
                values ({values})
                ;
            """

            stmt = (
                sql
                .SQL(query)
                .format(
                    table=sql.Identifier(table),
                    cols=sql.SQL(', ').join(map(sql.Identifier, cols)),
                    values=sql.SQL(', ').join(sql.Placeholder() * len(values[0]))
                )
            )

            # for v in values:
            cur.executemany(stmt, values)
            conn.commit()

            # ret = cur.fetchall()
            

except pg.OperationalError as e:
    logger.error(f"Connect to DB got error: {e}")

### `dim_meals`

In [ ]:
path = "data/processed/dim_meals.parquet"
dim_meals = pd.read_parquet(path)
dim_meals.head()

In [ ]:
values = [
    (
        r.id
        ,r.meal_codes.tolist()
        ,r.names.tolist()
        ,r.restaurants.tolist()
        ,r.meal_type
        ,r.schoolyear
        ,r.attributes.tolist()
        ,r.co2
        ,r.src.tolist()
    )
    for r in dim_meals.itertuples()
]

values[:2]

In [ ]:
# Compose SQL
cols = [
    'id',
    'meal_codes',
    'names',
    'restaurants',
    'meal_type',
    'schoolyear',
    'attributes',
    'co2',
    'src',
]
table = 'dim_meals'

try:
    with pg.connect(
        user=DB_USER,
        password=DB_PWD,
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        row_factory=dict_row,
    ) as conn:
        with conn.cursor() as cur:
            query = """
                insert into {table}
                    ({cols})
                values ({values})
                ;
            """

            stmt = (
                sql
                .SQL(query)
                .format(
                    table=sql.Identifier(table),
                    cols=sql.SQL(', ').join(map(sql.Identifier, cols)),
                    values=sql.SQL(', ').join(sql.Placeholder() * len(values[0]))
                )
            )

            # for v in values:
            cur.executemany(stmt, values)
            conn.commit()

            # ret = cur.fetchall()
            

except pg.OperationalError as e:
    logger.error(f"Connect to DB got error: {e}")

# Upload fact tables

### `menus`

In [ ]:
path_dir = Path("data/processed/planned_menu")

list_df = [pd.read_excel(path) for path in path_dir.glob("**/*/menus*.xlsx")]
df = pd.concat(list_df)

df.head()

In [ ]:
values = [
    (
        r.id,
        r.weeklevel_idx,
        r.restaurant,
        r.date,
        r.whole_pos,
        r.whole_waste,
        r.score,
    )
    for r in df.itertuples()
]

values[:2]

In [ ]:
cols = [
    'id',
    'weeklevel_idx',
    'restaurant',
    'date',
    'whole_pos',
    'whole_waste',
    'score',
]
table = 'menus'

try:
    with pg.connect(
        user=DB_USER,
        password=DB_PWD,
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        row_factory=dict_row,
    ) as conn:
        with conn.cursor() as cur:

            # Compose SQL

            query = """
                insert into {table}
                    ({cols})
                values ({values})
                ;
            """

            stmt = (
                sql
                .SQL(query)
                .format(
                    table=sql.Identifier(table),
                    cols=sql.SQL(', ').join(map(sql.Identifier, cols)),
                    values=sql.SQL(', ').join(sql.Placeholder() * len(values[0]))
                )
            )

            # for v in values:
            cur.executemany(stmt, values)
            conn.commit()

            # ret = cur.fetchall()
            

except pg.OperationalError as e:
    logger.error(f"Connect to DB got error: {e}")

### `meals_planned`

In [ ]:
path_dir = Path("data/processed/planned_menu")

list_df = [pd.read_excel(path) for path in path_dir.glob("**/*/meals_planned*.xlsx")]
df = pd.concat(list_df)

df.head()

In [ ]:
values = [
    (r.menu_id, r.meal, r.pos)
    for r in df.itertuples()
]

values[:2]

In [ ]:
# Compose SQL
cols = ['menu_id', 'meal', 'pos']
table = 'meals_planned'

try:
    with pg.connect(
        user=DB_USER,
        password=DB_PWD,
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        row_factory=dict_row,
    ) as conn:
        with conn.cursor() as cur:
            query = """
                insert into {table}
                    ({cols})
                values ({values})
                ;
            """

            stmt = (
                sql
                .SQL(query)
                .format(
                    table=sql.Identifier(table),
                    cols=sql.SQL(', ').join(map(sql.Identifier, cols)),
                    values=sql.SQL(', ').join(sql.Placeholder() * len(values[0]))
                )
            )

            # for v in values:
            cur.executemany(stmt, values)
            conn.commit()

            # ret = cur.fetchall()
            

except pg.OperationalError as e:
    logger.error(f"Connect to DB got error: {e}")